# 12 - Q1 Six-Arm Ablation Run

| Arm | Prompt | RAG | Validator | Model |
|---|---|---|---|---|
| A base_compact | compact | no | no | base |
| B base_rag | full | k=4 | no | base |
| C base_rag_val | full | k=4 | **yes** | base (reuses B raw bytes) |
| D ft_rag | full | k=4 | no | **fine-tuned** |
| E ft_rag_val | full | k=4 | **yes** | fine-tuned (reuses D raw bytes) |
| F ft_compact_val | compact | k=4 | yes | fine-tuned |

Pre-specified contrasts: **B-A** RAG - **C-B** validator (base) - **D-B** LoRA -
**E-D** validator (fine-tuned) - **F-E** compact-prompt trade-off.

Arms C and E never call a model: they load the stored raw text from B and D and
assert a SHA256 match, so the validator contribution is measured on identical
bytes by construction.

**Runtime:** GPU for arms A, B, D, F. Arms C and E are CPU-only.

Every record is appended immediately, so a interrupted run resumes without
regenerating completed outputs. Raw generations are never overwritten.


In [ ]:
# ============================================================
# Cell 1 - Environment and project code
# ============================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = ''                      # <-- set to your git remote, or upload the repo
REPO = Path('/content/Nephro-AI') if Path('/content').exists() else Path.cwd()

if not (REPO / 'llm').exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    else:
        raise SystemExit('Set REPO_URL or place the project at ' + str(REPO))

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('cwd:', Path.cwd())


In [ ]:
# ============================================================
# Cell 2 - Dependencies
# ============================================================
%pip install -q 'transformers>=4.44' 'peft>=0.12' 'accelerate>=0.33' 'sentence-transformers>=3.0' 'faiss-cpu>=1.8' 'scikit-learn==1.6.1' 'pandas==2.2.3' 'scipy==1.14.1' matplotlib

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} ({p.total_memory/1024**3:.1f} GB)')


## 1. Configuration

Point `ADAPTER` at the Phase-8 LoRA adapter directory (the folder containing
`adapter_config.json`). Arms D, E and F cannot run without it.


In [ ]:
# ============================================================
# Cell 3 - Configuration + benchmark integrity
# ============================================================
from llm.eval.q1 import run_arms as ra

BENCHMARK = Path('q1_benchmark/q1_profiles_300.jsonl')
RUNS      = Path('q1_runs')
ADAPTER   = 'llm/finetune/adapters/v1/adapter'   # <-- set to your adapter path
MODEL     = 'Qwen/Qwen3-4B-Instruct-2507'

assert BENCHMARK.exists(), f'Build the benchmark first (notebook 11): {BENCHMARK}'
profiles = ra.load_benchmark(BENCHMARK)
print('profiles      :', len(profiles))
print('benchmark hash:', ra.sha256_file(BENCHMARK))
print('adapter found :', Path(ADAPTER).exists(), '->', ADAPTER)


## 2. Primary run - deterministic decoding

`do_sample=False`, seed 42, identical max length, identical profile order,
no unplanned retry. Run the arms **in order** - C depends on B, E depends on D.


In [ ]:
# ============================================================
# Cell 4 - Arm A (base, compact, no RAG, no validator)
# ============================================================
ra.run_arm('A', BENCHMARK, RUNS, model_name=MODEL, seeds=[42], do_sample=False)


In [ ]:
# ============================================================
# Cell 5 - Arm B (base, full prompt, RAG k=4, no validator)
# ============================================================
ra.run_arm('B', BENCHMARK, RUNS, model_name=MODEL, seeds=[42], do_sample=False)


In [ ]:
# ============================================================
# Cell 6 - Arm C (validator applied to Arm B raw bytes; no generation)
# ============================================================
ra.run_arm('C', BENCHMARK, RUNS, seeds=[42])


In [ ]:
# ============================================================
# Cell 7 - Arm D (fine-tuned, full prompt, RAG k=4, no validator)
# ============================================================
ra.run_arm('D', BENCHMARK, RUNS, adapter=ADAPTER, model_name=MODEL,
           seeds=[42], do_sample=False)


In [ ]:
# ============================================================
# Cell 8 - Arm E (validator applied to Arm D raw bytes; no generation)
# ============================================================
ra.run_arm('E', BENCHMARK, RUNS, seeds=[42])


In [ ]:
# ============================================================
# Cell 9 - Arm F (proposed final system: fine-tuned + compact + RAG + validator)
# ============================================================
ra.run_arm('F', BENCHMARK, RUNS, adapter=ADAPTER, model_name=MODEL,
           seeds=[42], do_sample=False)


## 3. Secondary robustness (optional)

`do_sample=True`, temperature 0.2, top-p 0.9, seeds 42/52/62 on the **generative**
arms only. Profiles - not seeds - remain the statistical unit; the analysis
notebook aggregates within profile and uses profile-clustered bootstrap CIs.


In [ ]:
# ============================================================
# Cell 10 - Multi-seed robustness (optional, slow)
# ============================================================
RUN_MULTISEED = False    # set True to run the secondary analysis

if RUN_MULTISEED:
    for arm, ad in [('A', None), ('B', None), ('D', ADAPTER), ('F', ADAPTER)]:
        ra.run_arm(arm, BENCHMARK, Path('q1_runs_sampled'), adapter=ad,
                   model_name=MODEL, seeds=[42, 52, 62], do_sample=True,
                   temperature=0.2, top_p=0.9)
    for arm in ('C', 'E'):
        ra.run_arm(arm, BENCHMARK, Path('q1_runs_sampled'), seeds=[42, 52, 62])


## 4. Manifest and integrity


In [ ]:
# ============================================================
# Cell 11 - Manifest + PASS/FAIL integrity table
# ============================================================
import json
manifest = ra.write_manifest(RUNS, BENCHMARK)
print('manifest ->', manifest)

checks = []
for arm in ra.ARM_ORDER:
    p = ra._record_path(RUNS, arm)
    n = len(ra.load_done_keys(p)) if p.exists() else 0
    checks.append((f'arm_{arm}_records_300', n == 300, f'{n}/300'))
checks.extend(ra.verify_reuse(RUNS))

w = max(len(c[0]) for c in checks)
all_ok = True
print('\n=== RUN INTEGRITY TABLE ===')
for name, ok, detail in checks:
    all_ok &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  {name.ljust(w)}  {detail}")
print()
print('OVERALL:', 'PASS' if all_ok else 'FAIL')


Next: `13_ANALYZE_Q1_LLM_RESULTS.ipynb`.
